# 04 — Validação Interna das Hipóteses

## Objetivo

Validar **internamente** o indicador de qualificação da audiência construído no notebook anterior (`perfil_qualificacao`), verificando se os tiers se comportam de forma coerente com as expectativas teóricas. externa.

## Hipóteses
| Hipótese | Enunciado | Tipo |
|---|---|---|
| **H1** | Tiers superiores (A > B > C > D) apresentam **maior** ER clássico mediano. | Validade convergente |
| **H2** | Tiers superiores apresentam **menor** coeficiente de variação (CV) do ER. | Validade convergente |
| **H3** | Os resultados são **estáveis** quando se substitui `er_classico` por `er_weighted` na composição do score. | Robustez |
| **H4** | O padrão de ordenação dos tiers se mantém **dentro dos segmentos** (`inf_category × bucket_followers`). | Coerência intrassegmento |
| **H5** | A distribuição dos tiers não colapsa sistematicamente em um único porte de perfil, indicando **ausência de viés estrutural forte por bucket**. | Equilíbrio estrutural |


In [1]:

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr, kruskal

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 13

ORDEM_TIER = ["D", "C", "B", "A"]
ORDEM_BUCKET = ["nano", "micro", "mid-tier", "macro", "mega"]

# 1. Carga da base

In [ ]:
perfis = pl.read_parquet(Path("bases-processadas/perfil_qualificacao.parquet"))

print(perfis.shape)
perfis.head()

(33149, 21)


username,n_posts_total,followers,inf_category,bucket_followers,perfil_completo_lookup,erclassico_mediana,erclassico_media,erweighted_mediana,is_sponsored_pct,pct_image,pct_carousel,n_posts_validos,er_media,er_mediana,cv,elegivel,rank_er,rank_cv,score_audiencia,tier_audiencia
str,u32,f64,str,str,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64,str
"""camilomontoya""",300,91926.0,"""family""","""micro""",true,0.683702,0.899662,0.686422,0.0,0.99,0.01,300.0,0.899662,0.683702,0.772349,true,0.037618,0.850157,0.082508,"""D"""
"""marthashappilyeverafter""",300,39262.0,"""interior""","""micro""",true,3.914727,4.137427,3.928735,0.23,0.963333,0.036667,300.0,4.137427,3.914727,0.422518,true,0.838519,0.201481,0.822519,"""A"""
"""808projectd""",300,1280.0,"""other""","""nano""",true,1.25,1.393229,1.2890625,0.003333,0.893333,0.106667,300.0,1.393229,1.25,0.479176,true,0.250088,0.316251,0.423552,"""C"""
"""anna.rana""",200,14876.0,"""travel""","""micro""",true,1.317558,3.07102,1.317558,0.04,0.975,0.025,200.0,3.07102,1.317558,1.269942,true,0.117462,0.988464,0.075092,"""D"""
"""kidsneversleep""",300,26185.0,"""fashion""","""micro""",true,5.258736,5.527974,5.258736,0.093333,0.876667,0.123333,300.0,5.527974,5.258736,0.438352,true,0.778433,0.486726,0.67237,"""B"""


# 2. Sintese inicial

In [9]:
perfis = perfis.filter(pl.col("tier_audiencia") != "NaN")

In [ ]:
resumo_tiers = (
    perfis
    .group_by("tier_audiencia")
    .agg([
        pl.len().alias("n_perfis"),
        pl.col("erclassico_mediana").median().alias("erclassico_mediana_mediana"),
        pl.col("cv").median().alias("cv_mediana"),
        pl.col("score_audiencia").median().alias("score_mediano"),
    ])
    .with_columns((pl.col("n_perfis") / pl.col("n_perfis").sum() * 100).alias("pct"))
    .with_columns(
        pl.col("tier_audiencia")
        .replace({tier: i for i, tier in enumerate(ORDEM_TIER)})
        .alias("ordem_tier")
    )
    .sort("ordem_tier")
    .drop("ordem_tier")
)
resumo_tiers

tier_audiencia,n_perfis,erclassico_mediana_mediana,cv_mediana,score_mediano,pct
str,u32,f64,f64,f64,f64
"""D""",8253,1.274028,0.714366,0.2,25.002272
"""C""",8252,2.469168,0.5083,0.417678,24.999243
"""B""",8252,3.862057,0.456189,0.592467,24.999243
"""A""",8252,6.298462,0.348333,0.792385,24.999243
